In [ ]:
import json
import pandas as pd
import ipywidgets as widgets
from pathlib import Path
from typing import Dict, Any
from IPython.display import display, Markdown, JSON, HTML

# --- Configuration ---
RESULTS_DIR = Path("../data/results")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print(f"📂 Results Directory: {RESULTS_DIR.resolve()}")

In [ ]:
def _style_dataframe(df: pd.DataFrame) -> Any:
    """Applies visual styling to the experiments DataFrame."""
    # Remove empty columns
    df_clean = df.dropna(axis=1, how='all')
    
    format_dict = {
        'km_value': '{:.4f}', 'vmax_value': '{:.4f}',
        'ph': '{:.1f}', 'temperature': '{:.0f}',
        'c_min': '{:.2f}', 'c_max': '{:.2f}'
    }
    
    return df_clean.style.background_gradient(
        cmap='Blues', subset=['km_value', 'vmax_value']
    ).format(
        format_dict, na_rep="-"
    ).set_properties(**{
        'text-align': 'left', 'white-space': 'nowrap'
    })

def render_reasoning(text: str) -> None:
    """Renders the Chain-of-Thought reasoning block."""
    html = f"""
    <div style="background-color: #f8f9fa; border-left: 4px solid #3182ce; 
                padding: 12px; margin: 10px 0; border-radius: 4px; font-family: sans-serif;">
        <strong style="color: #2c5282;">🧠 Chain of Thought:</strong>
        <div style="margin-top: 8px; color: #2d3748; line-height: 1.5;">{text}</div>
    </div>
    """
    display(HTML(html))

def show_result(file_name: str) -> None:
    """Loads and displays analysis results for a selected file."""
    if not file_name:
        return

    file_path = RESULTS_DIR / file_name
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        display(Markdown(f"❌ **Error loading file:** {e}"))
        return

    # 1. Header
    meta = data.get("source_metadata", {})
    display(Markdown(f"# 📄 {meta.get('filename', file_name)}"))
    
    # 2. Reasoning
    if reasoning := data.get("reasoning"):
        render_reasoning(reasoning)

    # 3. Experiments Table
    experiments = data.get("extraction", {}).get("experiments", [])
    display(Markdown(f"### 🧪 Extracted Experiments ({len(experiments)})"))
    
    if experiments:
        display(_style_dataframe(pd.DataFrame(experiments)))
    else:
        display(Markdown("*No experiments found.*"))

    # 4. Raw JSON
    display(HTML("<details><summary style='cursor:pointer; color:#718096'>🔍 Raw JSON</summary>"))
    display(JSON(data, expanded=False))
    display(HTML("</details>"))

In [ ]:
# Scan directory
json_files = sorted([f.name for f in RESULTS_DIR.glob("*.json")])

if not json_files:
    print("⚠️ No result files found. Run 'scripts/predict.py' first.")
else:
    dropdown = widgets.Dropdown(
        options=json_files,
        description='Select File:',
        layout={'width': '600px'}
    )
    widgets.interact(show_result, file_name=dropdown);